# Lab 0 — HTTP Basics for FastAPI

Difficulty: Beginner | ~20-25 min | No prerequisites

### Step 1: Install Dependencies

We start by installing our target libraries: `fastapi` for the API framework and `httpx` as the HTTP client that powers FastAPI's `TestClient`.

In [ ]:
# Install the exact pinned versions of the required libraries.
!pip install fastapi==0.112.2 httpx==0.28.1


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

### Step 2: Python Imports

Next, we import the standard typing utility and the necessary classes from FastAPI that we will use to build the application.

In [1]:
# Import helpers for optional values, the app, the body, and the test client.
from typing import Optional
from fastapi import FastAPI, Body
from fastapi.testclient import TestClient

### Step 3: In-Memory Store

We keep a simple Python list as our "database". Each item is a plain dictionary — no Pydantic yet.

In [2]:
# A plain in-memory store. Restarting the kernel resets it.
items = [
    {"id": 1, "name": "Notebook", "price": 4.99},
    {"id": 2, "name": "Backpack", "price": 39.99},
]

### Step 4: GET Endpoint with a Query Parameter

We initialize the app and define a `GET` route that returns the whole list. `max_price` is *not* in the path, so FastAPI treats it as an optional **query parameter** (`/items?max_price=10`). Its default of `None` means the filter is skipped when the client does not send it.

Read the annotation piece by piece in the comments below: the name, the type, and the default each do a distinct job.

In [3]:
# Create the app and a GET route that can filter by query parameter.
app = FastAPI()

# max_price : Optional[float] = None
#   max_price         -> parameter name; becomes the "?max_price=" query key
#   : Optional[float] -> the value is a number (float) or nothing (None)
#   = None            -> a default, so the client may leave it out entirely
@app.get("/items")
def list_items(max_price: Optional[float] = None):
    if max_price is None:
        return items
    return [item for item in items if item["price"] <= max_price]

### Step 5: GET Endpoint with a Path Parameter

This route reads a single item. `{item_id}` is a **path parameter** — FastAPI takes it from the URL and converts it to an `int` because of the type hint. It returns the item whose `id` matches.

In [4]:
# A GET route that reads one item by its path parameter.
# item_id : int
#   item_id -> parameter name; must match the {item_id} in the route path
#   : int   -> FastAPI reads that part of the URL as an integer
@app.get("/items/{item_id}")
def get_item(item_id: int):
    for item in items:
        if item["id"] == item_id:
            return item

### Step 6: POST Endpoint with a Request Body

This route **creates** an item. The `item: dict = Body(...)` argument tells FastAPI to read the JSON body the client sends and pass it in as a dictionary. We assign the next `id`, store it, and return it. `status_code=201` sets the created status.

In [5]:
# A POST route that reads a JSON body, stores it, and returns it.
# item : dict = Body(...)
#   item        -> parameter name
#   : dict      -> the body arrives as a Python dictionary
#   = Body(...) -> tells FastAPI to read this from the JSON request body
@app.post("/items", status_code=201)
def create_item(item: dict = Body(...)):
    next_id = max(existing["id"] for existing in items) + 1
    item["id"] = next_id
    items.append(item)
    return item

### Step 7: Instantiate TestClient

Create a client wrapper to send requests to the app from inside the notebook.

In [6]:
# Instantiating TestClient to run local API requests.
client = TestClient(app)

### Step 8: Case 1 - GET the Collection

Reads every item.

In [7]:
# Case 1: Read the whole collection (expecting 200 OK).
response_1 = client.get("/items")
print("Status Code:", response_1.status_code)
print("Response JSON:", response_1.json())

Status Code: 200
Response JSON: [{'id': 1, 'name': 'Notebook', 'price': 4.99}, {'id': 2, 'name': 'Backpack', 'price': 39.99}]

### Step 9: Case 2 - GET with a Query Parameter

Appends `?max_price=10.0` to the URL so only cheaper items come back. The block below shows the query written directly into the URL, then the same request written with `params={...}` as an alternative.

In [8]:
# Case 2: Filter with a query parameter (expecting 200 OK).
# The query parameter is written straight into the URL after the "?".
response_2 = client.get("/items?max_price=10.0")
print("Status Code:", response_2.status_code)
print("Response JSON:", response_2.json())

# The same request can also be written with params={...} instead of the "?" string.
response_2b = client.get("/items", params={"max_price": 10.0})
print("Response JSON:", response_2b.json())

Status Code: 200
Response JSON: [{'id': 1, 'name': 'Notebook', 'price': 4.99}]
Response JSON: [{'id': 1, 'name': 'Notebook', 'price': 4.99}]

### Step 10: Case 3 - POST a New Item

Sends a JSON body to create a new item.

In [9]:
# Case 3: Create an item by sending a JSON body (expecting 201 Created).
new_item = {"name": "Water Bottle", "price": 12.50}
response_3 = client.post("/items", json=new_item)
print("Status Code:", response_3.status_code)
print("Response JSON:", response_3.json())

Status Code: 201
Response JSON: {'name': 'Water Bottle', 'price': 12.5, 'id': 3}

### Step 11: Case 4 - GET the New Item by Path Parameter

Reads back the item we just created, now that it has an assigned `id`.

In [10]:
# Case 4: Read the new item back by path parameter (expecting 200 OK).
response_4 = client.get("/items/3")
print("Status Code:", response_4.status_code)
print("Response JSON:", response_4.json())

Status Code: 200
Response JSON: {'name': 'Water Bottle', 'price': 12.5, 'id': 3}